# **Fama–French 5-Factor Model**
### **Factors:** `MKT` | `SMB` | `HML` | `RMW` | `CMA`

---

### **Overview**
- The **Fama–French 5-factor model** explains stock returns using **five systematic risk factors**
- It extends:
  - **CAPM**
  - **Fama–French 3-factor model**
- Adds:
  - **Profitability**
  - **Investment behavior**

---

### **Model Equation**
- **`Ri - Rf = α + βm(MKT) + βs(SMB) + βh(HML) + βr(RMW) + βc(CMA) + ε`**

---

### **Terms Explanation**
- **`Ri`** → Return of asset
- **`Rf`** → Risk-free rate
- **`MKT`** → Market return minus risk-free rate
- **`α` (Alpha)** → Excess return
- **`ε`** → Error term

---

### **Factor Definitions**

#### **1. `MKT` (Market Risk Premium)**
- **`MKT = Rm - Rf`**
- Captures overall market movement

---

#### **2. `SMB` (Small Minus Big)**
- Small-cap returns − Large-cap returns
- Proxy: Market capitalization

---

#### **3. `HML` (High Minus Low)**
- Value stocks − Growth stocks
- Proxy: Book-to-market ratio

---

#### **4. `RMW` (Robust Minus Weak)**
- High profitability − Low profitability
- Proxy: ROE, ROA

---

#### **5. `CMA` (Conservative Minus Aggressive)**
- Low investment − High investment
- Proxy: Asset growth

---

### **Intuition**
- Returns are driven by **systematic factor exposure**
- Each stock has **factor sensitivities (`β`)**
- These determine expected return

---

### **Why This Model Matters**
- Explains cross-sectional return differences
- Used in:
  - Portfolio construction
  - Quant investing
  - Asset pricing

---

### **Limitations**
- Linear assumptions
- Static factors
- Missing:
  - `Momentum`
  - Nonlinear effects

---

### **Project Connection**
- Construct all 5 factors from raw data
- Use them to:
  - Explain returns
  - Generate alpha
- Extend with ML

---

### **Expected Outputs**
- Factor series: `MKT`, `SMB`, `HML`, `RMW`, `CMA`
- Factor loadings
- Alpha
- Backtest results

### Phase 1: Data ingestion and Cleaning (Nifty 500: From 01-01-2011 till date)

In [1]:
# ==========================================
# Phase 1: Data Collection (NIFTY 100)
# ==========================================

from nselib import capital_market
import pandas as pd
from datetime import date, datetime
import time
import os
import requests
import feedparser
from urllib.parse import quote_plus
import shutil
from pathlib import Path
from transformers import pipeline


C:\Users\jayan\anaconda3\envs\ml_alpha_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# import sys
# !{sys.executable} -m pip install nselib
# !{sys.executable} -m pip install feedparser

In [3]:
# =====================================
# STEP 0: Project Setup
# =====================================

PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"

# Clean full data directory
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)

RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Clean data directories ready.")


Clean data directories ready.


In [4]:
FIN_PROCESSED_DIR = DATA_DIR / "fin_data_processed"
FIN_BACKUPS_DIR = DATA_DIR / "fin_data_backups"

FIN_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIN_BACKUPS_DIR.mkdir(parents=True, exist_ok=True)


In [5]:
# ====================================
# STEP 1: Fetch NIFTY 100 tickers
# ====================================

url = "https://archives.nseindia.com/content/indices/ind_nifty500list.csv"
df_list = pd.read_csv(url)

tickers = df_list["Symbol"].tolist()

print(f"Total tickers fetched: {len(tickers)}")


Total tickers fetched: 504


In [6]:
df_list.head()

,Company Name,Industry,Symbol,Series,ISIN Code
0,360 ONE WAM Ltd.,Financial Services,360ONE,EQ,INE466L01038
1,3M India Ltd.,Diversified,3MINDIA,EQ,INE470A01017
2,ABB India Ltd.,Capital Goods,ABB,EQ,INE117A01022
3,ACC Ltd.,Construction Materials,ACC,EQ,INE012A01025
4,ACME Solar Holdings Ltd.,Power,ACMESOLAR,EQ,INE622W01025


In [7]:
df_list.shape

(504, 5)

In [8]:
# ==================================
# STEP 2: Fetch Price + News
# ==================================

def fetch_all_data(symbols, start, end):

    failed = []

    for symbol in symbols:

        try:
            print(f"\n Fetching {symbol} ...")

            df = capital_market.price_volume_and_deliverable_position_data(
                symbol=symbol,
                from_date=start.strftime("%d-%m-%Y"),
                to_date=end.strftime("%d-%m-%Y"),
            )

            if df.empty:
                print(f" No price data for {symbol}")
                failed.append(symbol)
                continue

            df.columns = df.columns.str.lower().str.strip()

            # Keep only EQ series
            if "series" in df.columns:
                df = df[df["series"].str.upper() == "EQ"].copy()

                if df.empty:
                    print(f" {symbol} has no EQ data")
                    failed.append(symbol)
                    continue

            # Save price
            df.to_csv(RAW_DIR / f"{symbol}.csv", index=False)
            print(f" Price saved")

        except Exception as e:
            print(f"Failed: {symbol} | {e}")
            failed.append(symbol)

        time.sleep(1)

    print("\n Data Fetch Completed.")
    print(f" Success: {len(symbols) - len(failed)} | Failed: {len(failed)}")

    if failed:
        pd.DataFrame(failed, columns=["Symbol"]).to_csv(
            DATA_DIR / "failed_tickers.csv", index=False
        )
        print("Failed tickers saved.")

In [9]:
## =======================================================================
# STEP 3: Execute for retreiving stock price data of 500 stocks from 2011
# =======================================================================

START_DATE = date(2011, 1, 1)
END_DATE = date.today()

fetch_all_data(tickers, START_DATE, END_DATE)

print("Total RAW files:", len(list(RAW_DIR.glob("*.csv"))))


 Fetching 360ONE ...
 Price saved

 Fetching 3MINDIA ...
 Price saved

 Fetching ABB ...
 Price saved

 Fetching ACC ...
 Price saved

 Fetching ACMESOLAR ...
 Price saved

 Fetching AIAENG ...
 Price saved

 Fetching APLAPOLLO ...
 Price saved

 Fetching AUBANK ...
 Price saved

 Fetching AWL ...
 Price saved

 Fetching AADHARHFC ...
 Price saved

 Fetching AARTIIND ...
 Price saved

 Fetching AAVAS ...
 Price saved

 Fetching ABBOTINDIA ...
 Price saved

 Fetching ACE ...
 Price saved

 Fetching ACUTAAS ...
 Price saved

 Fetching ADANIENSOL ...
 Price saved

 Fetching ADANIENT ...
 Price saved

 Fetching ADANIGREEN ...
 Price saved

 Fetching ADANIPORTS ...
 Price saved

 Fetching ADANIPOWER ...
 Price saved

 Fetching ATGL ...
 Price saved

 Fetching ABCAPITAL ...
 Price saved

 Fetching ABFRL ...
 Price saved

 Fetching ABLBL ...
 Price saved

 Fetching ABREL ...
 Price saved

 Fetching ABSLAMC ...
 Price saved

 Fetching CPPLUS ...
 Price saved

 Fetching AEGISLOG ...
 Price sav

 Price saved

 Fetching ICICIGI ...
 Price saved

 Fetching ICICIAMC ...
 Price saved

 Fetching ICICIPRULI ...
 Price saved

 Fetching IDBI ...
 Price saved

 Fetching IDFCFIRSTB ...
 Price saved

 Fetching IFCI ...
 Price saved

 Fetching IIFL ...
 Price saved

 Fetching IRB ...
 Price saved

 Fetching IRCON ...
 Price saved

 Fetching ITCHOTELS ...
 Price saved

 Fetching ITC ...
 Price saved

 Fetching ITI ...
 Price saved

 Fetching INDGN ...
 Price saved

 Fetching INDIACEM ...
 Price saved

 Fetching INDIAMART ...
 Price saved

 Fetching INDIANB ...
 Price saved

 Fetching IEX ...
 Price saved

 Fetching INDHOTEL ...
 Price saved

 Fetching IOC ...
 Price saved

 Fetching IOB ...
 Price saved

 Fetching IRCTC ...
 Price saved

 Fetching IRFC ...
 Price saved

 Fetching IREDA ...
 Price saved

 Fetching IGL ...
 Price saved

 Fetching INDUSTOWER ...
 Price saved

 Fetching INDUSINDBK ...
 Price saved

 Fetching NAUKRI ...
 Price saved

 Fetching INFY ...
 Price saved

 Fetching I

 Price saved

 Fetching TECHM ...
 Price saved

 Fetching TECHNOE ...
 Price saved

 Fetching TEGA ...
 Price saved

 Fetching TEJASNET ...
 Price saved

 Fetching TENNIND ...
 Price saved

 Fetching NIACL ...
 Price saved

 Fetching RAMCOCEM ...
 Price saved

 Fetching THERMAX ...
 Price saved

 Fetching TIMKEN ...
 Price saved

 Fetching TITAGARH ...
 Price saved

 Fetching TITAN ...
 Price saved

 Fetching TORNTPHARM ...
 Price saved

 Fetching TORNTPOWER ...
 Price saved

 Fetching TARIL ...
 Price saved

 Fetching TRAVELFOOD ...
 Price saved

 Fetching TRENT ...
 Price saved

 Fetching TRIDENT ...
 Price saved

 Fetching TRITURBINE ...
 Price saved

 Fetching TIINDIA ...
 Price saved

 Fetching UCOBANK ...
 Price saved

 Fetching UNOMINDA ...
 Price saved

 Fetching UPL ...
 Price saved

 Fetching UTIAMC ...
 Price saved

 Fetching ULTRACEMCO ...
 Price saved

 Fetching UNIONBANK ...
 Price saved

 Fetching UBL ...
 Price saved

 Fetching UNITDSPR ...
 Price saved

 Fetching URBAN

In [10]:
# ======================================================
# STEP 4: Fetching Fundamental Data
# ======================================================

import warnings; warnings.filterwarnings("ignore")

folder_path = RAW_DIR
all_data = []

for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        temp = pd.read_csv(os.path.join(folder_path, file), encoding='latin1')
        
        temp.columns = temp.columns.str.strip().str.lower()
        temp['symbol'] = file.replace(".csv", "")
        
        # Clean price
        temp['closeprice'] = temp['closeprice'].astype(str).str.replace(',', '')
        temp['closeprice'] = pd.to_numeric(temp['closeprice'], errors='coerce')
        
        # FIXED DATE PARSING
        temp['date'] = pd.to_datetime(temp['date'], dayfirst=True, errors='coerce')
        
        # Drop invalid dates
        temp = temp.dropna(subset=['date'])
        
        temp = temp.sort_values('date')
        temp.set_index('date', inplace=True)
        
        # Monthly resample
        monthly = temp.resample('M').last().reset_index()
        monthly['return'] = monthly['closeprice'].pct_change()
        
        monthly['year'] = monthly['date'].dt.year
        monthly['month'] = monthly['date'].dt.month
        
        monthly['effective_year'] = monthly['year']
        monthly.loc[monthly['month'] < 7, 'effective_year'] -= 1
        
        all_data.append(monthly)

price_data = pd.concat(all_data, ignore_index=True)
price_data = price_data.sort_values('date').reset_index(drop=True)

In [11]:
price_data.sort_values('date')

,date,"ã¯â»â¿""symbol""",series,prevclose,openprice,highprice,lowprice,lastprice,closeprice,averageprice,totaltradedquantity,turnoverinrs,no.oftrades,deliverableqty,%dlyqttotradedqty,symbol,return,year,month,effective_year
0,2011-01-31,RKFORGE,EQ,108.95,120.0,120.0,102.2,110.0,115.35,115.28,10504.0,1.210903e+06,NaN,9192.0,87.51,RKFORGE,NaN,2011,1,2010
180,2011-01-31,IDBI,EQ,141.35,139.2,143.2,136.25,141.95,141.75,140.1,7445788.0,1.043138e+09,NaN,1487674.0,19.98,IDBI,NaN,2011,1,2010
181,2011-01-31,FACT,EQ,44.4,44.0,48.5,42.3,45.6,45.00,43.88,35162.0,1.542900e+06,NaN,13065.0,37.16,FACT,NaN,2011,1,2010
182,2011-01-31,OBEROIRLTY,EQ,245.75,247.0,250.95,245.0,249.0,247.55,247.58,27313.0,6.762123e+06,NaN,19293.0,70.64,OBEROIRLTY,NaN,2011,1,2010
183,2011-01-31,SCI,EQ,117.6,117.0,117.0,113.85,114.45,114.25,115.14,512972.0,5.906530e+07,NaN,314918.0,61.39,SCI,NaN,2011,1,2010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67554,2026-05-31,LTTS,EQ,"3,772.90","3,795.00","3,847.90","3,774.10","3,816.00",3802.60,"3,807.72",88953.0,3.387083e+08,17529.0,25568.0,28.74,LTTS,0.048617,2026,5,2025
67553,2026-05-31,ONGC,EQ,283.9,283.6,284.0,278.6,279.3,279.20,280.31,9290168.0,2.604141e+09,73595.0,4504834.0,48.49,ONGC,-0.067935,2026,5,2025
67552,2026-05-31,SPLPETRO,EQ,741.0,744.35,749.75,732.65,738.4,739.90,740.74,101186.0,7.495227e+07,4272.0,71467.0,70.63,SPLPETRO,-0.022976,2026,5,2025
67550,2026-05-31,SBIN,EQ,"1,092.00","1,090.00","1,095.00","1,010.90","1,018.40",1019.30,"1,037.45",48065909.0,4.986589e+10,592384.0,21794655.0,45.34,SBIN,-0.046001,2026,5,2025


In [12]:
# Total number of monthly periods
total_periods = price_data['date'].nunique()

# Keep stocks with at least 70% data
threshold = 0.7 * total_periods

valid_stocks = (
    price_data.groupby('symbol')['closeprice']
    .count()
)

valid_stocks = valid_stocks[valid_stocks > threshold].index

price_data = price_data[price_data['symbol'].isin(valid_stocks)].copy()

In [13]:
median_vol = (
    price_data.groupby('symbol')['totaltradedquantity']
    .median()
)

liquid_stocks = median_vol[median_vol > 1e5].index

price_data = price_data[price_data['symbol'].isin(liquid_stocks)].copy()

In [14]:
price_data['totaltradedquantity'] = pd.to_numeric(
    price_data['totaltradedquantity'],
    errors='coerce'
)

In [15]:
price_data['turnover'] = (
    price_data['closeprice'] * price_data['totaltradedquantity']
)

In [17]:
# =====================================================================
# STEP 5: Fetching Risk-free returns (91-day Govt. Treasury bills)
# =====================================================================

url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=INDIR3TIB01STM"

df = pd.read_csv(url)
df['DATE'] = pd.to_datetime(df['observation_date'])

df = df.set_index('DATE')
df.rename(columns={'INDIR3TIB01STM': 'Rf'}, inplace=True) #Rf means risk-free returns

df['Rf'] = df['Rf'] / 100
df = df.resample('D').ffill()
df['Rf_daily'] = df['Rf'] / 365

df = df[['Rf_daily']].copy()

df.to_csv(DATA_DIR / "rf_india.csv")

In [18]:
rf = pd.read_csv(DATA_DIR / "rf_india.csv", parse_dates=['DATE'])
rf = rf.set_index('DATE')

rf_monthly = rf.resample('M').apply(lambda x: (1 + x).prod() - 1)
rf_monthly = rf_monthly.rename(columns={'Rf_daily': 'Rf_monthly'})

In [19]:
# =====================================================================
# STEP 6: Financial Data Pipeline
# =====================================================================

In [20]:
import yfinance as yf
import requests
import pandas as pd
import numpy as np
import time, re, json
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def create_session():
    s = requests.Session()
    s.mount("https://", HTTPAdapter(max_retries=Retry(
        total=3, backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504])))
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "application/json, */*",
    })
    return s


In [21]:
symbols = tickers

In [22]:
# =============================================================================
# SOURCE 1 — yfinance with the correct API calls (not .quarterly_financials)
# Gets ~2015 onwards for most Nifty500 stocks
# =============================================================================

def fetch_yfinance(symbol):
    """
    Uses get_income_stmt / get_balance_sheet instead of the
    .quarterly_financials property — these hit a different Yahoo endpoint
    that returns more history (typically 12-16 quarters vs 4-5).
    """
    try:
        ticker = yf.Ticker(f"{symbol}.NS")

        # These methods return more history than the .quarterly_financials property
        inc = ticker.get_income_stmt(freq="quarterly", as_dict=False)
        bal = ticker.get_balance_sheet(freq="quarterly", as_dict=False)

        if inc is None or inc.empty:
            # Final fallback to the property
            inc = ticker.quarterly_financials
        if bal is None or bal.empty:
            bal = ticker.quarterly_balance_sheet

        if inc is None or inc.empty or bal is None or bal.empty:
            return None, "yf_empty"

        NI_KEYS = [
            "Net Income",
            "Net Income From Continuing Operations",
            "Net Income Including Noncontrolling Interests",
            "Net Income Common Stockholders",
        ]
        TA_KEYS = ["Total Assets"]

        ni_row = next((inc.loc[k] for k in NI_KEYS if k in inc.index), None)
        ta_row = next((bal.loc[k] for k in TA_KEYS if k in bal.index), None)

        if ni_row is None: return None, "yf_no_net_income"
        if ta_row is None: return None, "yf_no_assets"

        all_dates = sorted(
            set(ni_row.dropna().index) | set(ta_row.dropna().index)
        )

        records = []
        for dt in all_dates:
            month = dt.month
            if   month in [4,5,6]:   qtr, fy = "Q1", dt.year + 1
            elif month in [7,8,9]:   qtr, fy = "Q2", dt.year + 1
            elif month in [10,11,12]: qtr, fy = "Q3", dt.year + 1
            else:                     qtr, fy = "Q4", dt.year

            ni_val = ni_row.get(dt)
            ta_val = ta_row.get(dt)

            if pd.isna(ni_val) and pd.isna(ta_val):
                continue

            records.append({
                "year":         fy,
                "quarter":      qtr,
                "net_income":   float(ni_val) / 1e7 if pd.notna(ni_val) else None,
                "total_assets": float(ta_val) / 1e7 if pd.notna(ta_val) else None,
                "source":       "yfinance",
            })

        if not records:
            return None, "yf_no_records"

        df = (pd.DataFrame(records)
              .drop_duplicates(subset=["year", "quarter"])
              .sort_values(["year", "quarter"])
              .reset_index(drop=True))
        return df, None

    except Exception as e:
        return None, f"yf_error: {e}"


In [23]:
# =============================================================================
# SOURCE 2 — Screener.in annual data (HTML table scrape, NOT the JSON API)
# Gets 2014-2015 onwards reliably, annual only
# The HTML tables are far more stable than the JSON endpoint
# =============================================================================

from bs4 import BeautifulSoup

def fetch_screener_html(symbol, session):
    """
    Scrapes annual P&L and Balance Sheet from Screener.in HTML.
    
    KEY FIX: parse_table now keeps the FIRST non-empty occurrence of each
    row label. Screener's HTML contains duplicate labels (e.g. "Reserves"
    appears as a real data row, then again as an empty section-header row).
    The original dict-assignment silently overwrote real data with empty
    strings, causing book_value = None for every stock.
    """
 
    urls = [
        f"https://www.screener.in/company/{symbol}/consolidated/?view=10years",
        f"https://www.screener.in/company/{symbol}/?view=10years",
    ]
 
    # ------------------------------------------------------------------ #
    # FIXED parse_table: first non-empty occurrence of each label wins    #
    # ------------------------------------------------------------------ #
    def parse_table(tbl):
        rows = {}
        headers = []
        EMPTY_VALS = {'', '-', 'na', 'n/a', 'nan', '--'}
 
        for i, row in enumerate(tbl.find_all("tr")):
            cells = [c.get_text(strip=True) for c in row.find_all(["th", "td"])]
            if not cells:
                continue
 
            if i == 0:
                headers = cells
                continue
 
            label  = cells[0].strip()
            values = cells[1:]
 
            if not label:
                continue
 
            def row_has_data(vals):
                return any(
                    v.strip().lower() not in EMPTY_VALS for v in vals
                )
 
            if label not in rows:
                rows[label] = values
            else:
                # Only overwrite if stored value is all-empty AND new row has data.
                # First real data row always wins.
                if not row_has_data(rows[label]) and row_has_data(values):
                    rows[label] = values
 
        return headers, rows
 
    # ------------------------------------------------------------------ #
    # Safe value extractor                                                 #
    # ------------------------------------------------------------------ #
    def get_val(row_vals, col_idx):
        try:
            v = str(row_vals[col_idx]).replace(",", "").replace("(", "-").replace(")", "").strip()
            if v.lower() in {'', '-', 'na', 'n/a', 'nan', '--', 'ttm'}:
                return None
            return float(v)
        except (IndexError, ValueError, TypeError):
            return None
 
    for url in urls:
        try:
            res = session.get(url, timeout=15)
            if res.status_code != 200:
                continue
            if len(res.text) < 5000 or "No data" in res.text:
                continue
 
            soup = BeautifulSoup(res.text, "html.parser")
            sections = soup.find_all("section")
 
            pnl_table, bal_table = None, None
 
            for sec in sections:
                heading = sec.find("h2") or sec.find("h3")
                if not heading:
                    continue
                h = heading.text.lower().strip()
                table = sec.find("table")
                if not table:
                    continue
 
                if ("profit" in h or "income" in h or "p&l" in h) and pnl_table is None:
                    pnl_table = table
                elif "balance" in h and bal_table is None:
                    bal_table = table
 
            if pnl_table is None or bal_table is None:
                continue
 
            pnl_headers, pnl_rows = parse_table(pnl_table)
            bal_headers, bal_rows = parse_table(bal_table)
 
            # -------------------------------------------------------------- #
            # Year-to-column-index maps (built independently per table)       #
            # -------------------------------------------------------------- #
            def build_year_index(headers):
                result = []
                for idx, h in enumerate(headers[1:], start=0):
                    m = re.search(r'(20\d{2})', h)
                    if m:
                        result.append((idx, m.group(1)))
                return result
 
            pnl_year_indices = build_year_index(pnl_headers)
            bal_year_indices = build_year_index(bal_headers)
 
            if not pnl_year_indices:
                continue
 
            # Map: year_string -> col_index, for the balance sheet
            bal_year_map = {yr: idx for idx, yr in bal_year_indices}
 
            # -------------------------------------------------------------- #
            # Locate net income row (P&L)                                     #
            # -------------------------------------------------------------- #
            NI_RE = re.compile(r"net profit|profit after tax|\bpat\b", re.I)
            ni_vals_raw = None
            for label, vals in pnl_rows.items():
                if NI_RE.search(label):
                    ni_vals_raw = vals
                    break
 
            # -------------------------------------------------------------- #
            # Locate total assets row (Balance Sheet)                         #
            # -------------------------------------------------------------- #
            ta_vals_raw = None
            for label, vals in bal_rows.items():
                if "total assets" in label.lower().strip():
                    ta_vals_raw = vals
                    break
 
            # -------------------------------------------------------------- #
            # Locate book value components (Balance Sheet)                    #
            #                                                                 #
            # Strategy (in priority order):                                   #
            #   1. Direct equity row: "Total Equity", "Shareholders' Equity", #
            #      "Net Worth", "Total Stockholders Equity" (IND-AS single)   #
            #   2. Computed: Share Capital + Reserves                         #
            # -------------------------------------------------------------- #
 
            DIRECT_EQUITY_RE = re.compile(
                r"total equity|shareholders['\s]*equity|net worth|"
                r"stockholders['\s]*equity|total networth|equity \(net worth\)",
                re.I
            )
 
            SC_RE = re.compile(
                r"equity capital|equity share capital|share capital",
                re.I
            )
            RES_RE = re.compile(
                r"\breserves\b|other equity|surplus",
                re.I
            )
 
            direct_equity_raw = None
            share_capital_raw = None
            reserve_raw       = None
 
            for label, vals in bal_rows.items():
                if direct_equity_raw is None and DIRECT_EQUITY_RE.search(label):
                    direct_equity_raw = vals
                if share_capital_raw is None and SC_RE.search(label):
                    share_capital_raw = vals
                if reserve_raw is None and RES_RE.search(label):
                    reserve_raw = vals
 
            # -------------------------------------------------------------- #
            # Build records                                                    #
            # -------------------------------------------------------------- #
            records = []
            for pnl_col_idx, yr in pnl_year_indices:
 
                # P&L column uses pnl_col_idx directly
                ni = get_val(ni_vals_raw, pnl_col_idx) if ni_vals_raw is not None else None
 
                # Balance sheet column: look up by year, fall back to same index
                bal_col_idx = bal_year_map.get(yr, pnl_col_idx)
 
                ta = get_val(ta_vals_raw, bal_col_idx) if ta_vals_raw is not None else None
 
                # Book value: try direct equity first, then computed
                bv = None
                if direct_equity_raw is not None:
                    bv = get_val(direct_equity_raw, bal_col_idx)
 
                if bv is None and share_capital_raw is not None and reserve_raw is not None:
                    sc = get_val(share_capital_raw, bal_col_idx)
                    rv = get_val(reserve_raw, bal_col_idx)
                    if sc is not None and rv is not None:
                        bv = sc + rv
 
                records.append({
                    "year":         int(yr),
                    "quarter":      "FY",
                    "net_income":   ni,
                    "total_assets": ta,
                    "book_value":   bv,
                    "source":       "screener_html",
                })
 
            if not records:
                continue
 
            df = (
                pd.DataFrame(records)
                .drop_duplicates("year")
                .sort_values("year")
                .reset_index(drop=True)
            )
            return df, None
 
        except Exception as e:
            print(f"[Screener ERROR] {symbol}: {e}")
            continue
 
        time.sleep(1.5)
 
    return None, "screener_html_failed"

In [24]:
# =============================================================================
# SOURCE 3 — NSE India quarterly results (XBRL XML, publicly accessible)
# More reliable than BSE for unauthenticated requests
# Gets 2014 onwards quarterly for most stocks
# =============================================================================

def fetch_nse_quarterly(symbol, session):
    """
    NSE publishes quarterly result XMLs via their public archive.
    No authentication required. Endpoint is stable.
    """
    url = (
        f"https://www.nseindia.com/api/results-comparisons?"
        f"index=equities&symbol={symbol}"
    )
    try:
        # NSE requires these specific headers
        nse_session = requests.Session()
        nse_session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
            "Accept": "application/json",
            "Referer": "https://www.nseindia.com/",
            "Accept-Language": "en-US,en;q=0.9",
        })
        # NSE requires a cookie — get it first with a homepage visit
        nse_session.get("https://www.nseindia.com", timeout=10)
        time.sleep(1)

        res = nse_session.get(url, timeout=15)
        if res.status_code != 200:
            return None, f"nse_status_{res.status_code}"

        data = res.json()
        rows = data.get("data", [])
        if not rows:
            return None, "nse_no_data"

        records = []
        for row in rows:
            period = row.get("period", "")          # e.g. "Q2 FY2023"
            ni  = row.get("net_profit_loss")
            ta  = row.get("total_assets")

            qm = re.search(r'(Q[1-4])\s*FY(\d{2,4})', period, re.I)
            if qm:
                qtr = qm.group(1).upper()
                fy  = int(qm.group(2))
                fy  = 2000 + fy if fy < 100 else fy
            else:
                continue

            try:
                records.append({
                    "year":         fy,
                    "quarter":      qtr,
                    "net_income":   float(str(ni).replace(",","")) if ni else None,
                    "total_assets": float(str(ta).replace(",","")) if ta else None,
                    "source":       "nse_api",
                })
            except (ValueError, TypeError):
                continue

        if not records:
            return None, "nse_no_parseable_rows"

        df = (pd.DataFrame(records)
              .drop_duplicates(subset=["year","quarter"])
              .sort_values(["year","quarter"])
              .reset_index(drop=True))
        return df, None

    except Exception as e:
        return None, f"nse_error: {e}"


In [25]:
# =============================================================================
# MASTER PIPELINE
# =============================================================================

def run_pipeline(symbol, session):
    results = []

    # Source 1: yfinance (~2015 onwards, quarterly)
    df, err = fetch_yfinance(symbol)
    if df is not None and len(df) > 0:
        df.insert(0, "symbol", symbol)
        results.append(df)
        print(f"  yf: {len(df)} qtrs ({df['year'].min()}-{df['year'].max()})", end="")

    # Source 2: Screener HTML (~2014 onwards, annual)
    df, err = fetch_screener_html(symbol, session)
    if df is not None and len(df) > 0:
        df.insert(0, "symbol", symbol)
        results.append(df)
        print(f"  scr: {len(df)} yrs", end="")

    # Source 3: NSE API (~2014 onwards, quarterly)
    df, err = fetch_nse_quarterly(symbol, session)
    if df is not None and len(df) > 0:
        df.insert(0, "symbol", symbol)
        results.append(df)
        print(f"  nse: {len(df)} qtrs", end="")

    if not results:
        return None, "all_sources_failed"

    # Merge all sources — quarterly beats annual for same period
    # Priority: yfinance > nse_api > screener_html
    combined = pd.concat(results, ignore_index=True)
    
    # --- Extract book_value separately BEFORE deduplication ---
    # Screener carries book_value; yfinance/NSE do not.
    # Deduplication (yfinance wins) would discard Screener rows and lose book_value.
    bv_df = (
        combined[combined["book_value"].notna()][["symbol", "year", "quarter", "book_value"]]
        .drop_duplicates(subset=["symbol", "year"])
        .copy()
    ) if "book_value" in combined.columns else pd.DataFrame()

    # Deduplicate financial rows by source priority (yfinance > nse_api > screener_html)
    priority = {"yfinance": 0, "nse_api": 1, "screener_html": 2}
    combined["_rank"] = combined["source"].map(priority).fillna(9)
    combined = (combined
        .sort_values("_rank")
        .drop_duplicates(subset=["symbol", "year", "quarter"], keep="first")
        .drop(columns="_rank")
        .sort_values(["year", "quarter"])
        .reset_index(drop=True))

    # --- Re-attach book_value after dedup ---
    if not bv_df.empty:
        if "book_value" in combined.columns:
            combined = combined.drop(columns=["book_value"])
        combined = combined.merge(
            bv_df[["symbol", "year", "book_value"]].drop_duplicates(subset=["symbol", "year"]),
            on=["symbol", "year"],
            how="left"
        )

    return combined, None

In [26]:
# =============================================================================
# EXECUTION
# =============================================================================

session = create_session()
all_data, failures = [], {}

for i, sym in enumerate(symbols):
    print(f"\n[{i+1}/{len(symbols)}] {sym}", end="  ")

    df, err = run_pipeline(sym, session)

    if df is not None:
        df["net_income"]   = pd.to_numeric(df["net_income"],   errors="coerce")
        df["total_assets"] = pd.to_numeric(df["total_assets"], errors="coerce")
        df = df[df["total_assets"] > 0].dropna(subset=["net_income"])

        if len(df) > 0:
            all_data.append(df)
            print(f"  → {len(df)} total rows")
        else:
            failures[sym] = "all_rows_invalid"
            print(f"  → all rows invalid after cleaning")
    else:
        failures[sym] = err
        print(f"  → FAILED: {err}")

    time.sleep(1.5 if (i+1) % 20 != 0 else 5.0)

    if (i+1) % 50 == 0 and all_data:
        pd.concat(all_data, ignore_index=True).to_csv(FIN_BACKUPS_DIR / f"fundamentals_backup_{i+1}.csv")
        print(f"\n  [backup saved at step {i+1}]")


final_df = pd.concat(all_data, ignore_index=True)

# Debug: confirm book_value presence
print("Columns in final_df:", final_df.columns.tolist())
print("book_value non-null count:", final_df['book_value'].notna().sum() if 'book_value' in final_df.columns else "COLUMN MISSING")
screener_rows = final_df[final_df['source'] == 'screener_html'] if 'source' in final_df.columns else pd.DataFrame()
print("Screener rows:", len(screener_rows))
if not screener_rows.empty and 'book_value' in screener_rows.columns:
    print("Screener book_value non-null:", screener_rows['book_value'].notna().sum())

if "book_value" not in final_df.columns:
    final_df["book_value"] = np.nan

# ==========================================================
# CLEAN BOOK VALUE PIPELINE
# ==========================================================

# Convert safely
final_df['book_value'] = pd.to_numeric(
    final_df['book_value'],
    errors='coerce'
)

# Keep only required columns
book_df = final_df[
    ['symbol', 'year', 'quarter', 'book_value']
].copy()

# After book_df is built, verify year alignment
print("book_df year sample:\n", book_df[['symbol','year','quarter','book_value']].head(10))

# Remove missing values
book_df = book_df.dropna(subset=['book_value'])

print("Initial non-null BV rows:", len(book_df))

# Keep annual values
book_df = book_df[
    book_df['quarter'].isin(['FY', 'Q4', 'Q3', 'Q2', 'Q1'])
]

print("Annual BV rows:", len(book_df))

# Create fiscal year-end date
book_df['date'] = pd.to_datetime(
    book_df['year'].astype(str) + "-03-31"
)

# Remove duplicates
book_df = (
    book_df
    .sort_values(['symbol', 'date'])
    .drop_duplicates(['symbol', 'date'], keep='last')
)

# Convert Cr → ₹
book_df['book_value'] = book_df['book_value'] * 1e7

# ==========================================================
# Monthly expansion
# ==========================================================

monthly_frames = []

for sym in book_df['symbol'].unique():

    temp = (
        book_df[book_df['symbol'] == sym]
        .sort_values('date')
        .set_index('date')[['book_value']]
        .resample('M')
        .ffill()
        .reset_index()
    )

    temp['symbol'] = sym

    monthly_frames.append(temp)

# Safe concat
if len(monthly_frames) > 0:

    book_monthly = pd.concat(
        monthly_frames,
        ignore_index=True
    )

else:
    print("WARNING: book_monthly is empty. Diagnostics:")
    print(f"  book_df shape: {book_df.shape}")
    print(f"  final_df book_value non-null: {final_df['book_value'].notna().sum()}")
    raise ValueError(
        "book_monthly is empty — check that Screener is returning book_value "
        "and that deduplication is not dropping all Screener rows."
    )
    
# Lag to avoid lookahead bias
book_monthly['book_value'] = (
    book_monthly
    .groupby('symbol')['book_value']
    .shift(3)
)

print(
    "Book monthly non-null:",
    book_monthly['book_value'].notna().sum()
)

final_df = final_df[final_df["year"].astype(int) >= 2011].reset_index(drop=True)
final_df = final_df.sort_values(["symbol","year","quarter"]).reset_index(drop=True)

pd.DataFrame(list(failures.items()), columns=["symbol","reason"]).to_csv(FIN_BACKUPS_DIR / "failures.csv", index=False)

print(f"\nSymbols     : {final_df['symbol'].nunique()}")
print(f"Total rows  : {len(final_df)}")
print(f"Year range  : {final_df['year'].min()} → {final_df['year'].max()}")
print(f"Avg rows/co : {len(final_df)/final_df['symbol'].nunique():.1f}")
print(f"Failures    : {len(failures)}")
print(f"\nSource breakdown:\n{final_df['source'].value_counts()}")


[1/504] 360ONE    scr: 9 yrs  → 9 total rows

[2/504] 3MINDIA    scr: 7 yrs  → 7 total rows

[3/504] ABB    scr: 4 yrs  → 4 total rows

[4/504] ACC    scr: 12 yrs  → 12 total rows

[5/504] ACMESOLAR    scr: 7 yrs  → 7 total rows

[6/504] AIAENG    scr: 12 yrs  → 12 total rows

[7/504] APLAPOLLO    scr: 12 yrs  → 12 total rows

[8/504] AUBANK    scr: 4 yrs  → 4 total rows

[9/504] AWL    scr: 8 yrs  → 8 total rows

[10/504] AADHARHFC    scr: 8 yrs  → 8 total rows

[11/504] AARTIIND    scr: 12 yrs  → 12 total rows

[12/504] AAVAS    scr: 7 yrs  → 7 total rows

[13/504] ABBOTINDIA    scr: 12 yrs  → 12 total rows

[14/504] ACE    scr: 12 yrs  → 12 total rows

[15/504] ACUTAAS    scr: 10 yrs  → 10 total rows

[16/504] ADANIENSOL    scr: 12 yrs  → 12 total rows

[17/504] ADANIENT    scr: 12 yrs  → 12 total rows

[18/504] ADANIGREEN    scr: 11 yrs  → 11 total rows

[19/504] ADANIPORTS    scr: 12 yrs  → 12 total rows

[20/504] ADANIPOWER    scr: 12 yrs  → 12 total rows

[21/504] ATGL    scr: 

[162/504] ENGINERSIN    scr: 12 yrs  → 12 total rows

[163/504] ERIS    scr: 12 yrs  → 12 total rows

[164/504] ESCORTS    scr: 12 yrs  → 12 total rows

[165/504] ETERNAL    scr: 9 yrs  → 9 total rows

[166/504] EXIDEIND    scr: 12 yrs  → 12 total rows

[167/504] NYKAA    scr: 8 yrs  → 8 total rows

[168/504] FEDERALBNK    scr: 12 yrs  → 12 total rows

[169/504] FACT    scr: 11 yrs  → 11 total rows

[170/504] FINCABLES    scr: 10 yrs  → 10 total rows

[171/504] FSL    scr: 12 yrs  → 12 total rows

[172/504] FIVESTAR    scr: 5 yrs  → 5 total rows

[173/504] FORCEMOT    scr: 12 yrs  → 12 total rows

[174/504] FORTIS    scr: 12 yrs  → 12 total rows

[175/504] GAIL    scr: 12 yrs  → 12 total rows

[176/504] GVT&D    scr: 1 yrs  → 1 total rows

[177/504] GMRAIRPORT    scr: 12 yrs  → 12 total rows

[178/504] GABRIEL    scr: 2 yrs  → 2 total rows

[179/504] GALLANTT    scr: 12 yrs  → 12 total rows

[180/504] GRSE    scr: 12 yrs  → 12 total rows

[181/504] GICRE    scr: 12 yrs  → 12 total rows

[323/504] MAXHEALTH    scr: 9 yrs  → 9 total rows

[324/504] MAZDOCK    scr: 12 yrs  → 12 total rows

[325/504] MEESHO    → FAILED: all_sources_failed

[326/504] MINDACORP    scr: 12 yrs  → 12 total rows

[327/504] MSUMI    scr: 6 yrs  → 6 total rows

[328/504] MOTILALOFS    scr: 12 yrs  → 12 total rows

[329/504] MPHASIS    scr: 12 yrs  → 12 total rows

[330/504] MCX    scr: 12 yrs  → 12 total rows

[331/504] MUTHOOTFIN    scr: 11 yrs  → 11 total rows

[332/504] NATCOPHARM    scr: 12 yrs  → 12 total rows

[333/504] NBCC    scr: 12 yrs  → 12 total rows

[334/504] NCC    scr: 12 yrs  → 12 total rows

[335/504] NHPC    scr: 12 yrs  → 12 total rows

[336/504] NLCINDIA    scr: 12 yrs  → 12 total rows

[337/504] NMDC    scr: 12 yrs  → 12 total rows

[338/504] NSLNISP    scr: 6 yrs  → 4 total rows

[339/504] NTPCGREEN    scr: 3 yrs  → 3 total rows

[340/504] NTPC    scr: 12 yrs  → 12 total rows

[341/504] NH    scr: 12 yrs  → 12 total rows

[342/504] NATIONALUM    scr: 12 yrs  → 12 total row

[483/504] USHAMART    scr: 12 yrs  → 12 total rows

[484/504] VTL    scr: 12 yrs  → 12 total rows

[485/504] VBL    scr: 12 yrs  → 12 total rows

[486/504] VEDL    scr: 12 yrs  → 12 total rows

[487/504] VIJAYA    scr: 9 yrs  → 9 total rows

[488/504] VMM    scr: 6 yrs  → 6 total rows

[489/504] IDEA    scr: 12 yrs  → 12 total rows

[490/504] VOLTAS    scr: 12 yrs  → 12 total rows

[491/504] WAAREEENER    scr: 7 yrs  → 7 total rows

[492/504] WELCORP    scr: 12 yrs  → 12 total rows

[493/504] WELSPUNLIV    scr: 12 yrs  → 12 total rows

[494/504] WHIRLPOOL    scr: 8 yrs  → 8 total rows

[495/504] WIPRO    scr: 12 yrs  → 12 total rows

[496/504] WOCKPHARMA    scr: 12 yrs  → 12 total rows

[497/504] YESBANK    scr: 12 yrs  → 12 total rows

[498/504] ZFCVINDIA    scr: 4 yrs  → 4 total rows

[499/504] ZEEL    scr: 12 yrs  → 12 total rows

[500/504] ZENTEC    scr: 12 yrs  → 12 total rows

  [backup saved at step 500]

[501/504] ZENSARTECH    scr: 12 yrs  → 12 total rows

[502/504] ZYDUSLIFE 

In [27]:
# Got errors several times while fetching the data, so running diagnostics for a single company to check the tags

"""
Run this as a STANDALONE cell in your notebook BEFORE the main pipeline.
It will print the exact HTML structure Screener returns for one stock,
so we can see exactly what parse_table is working with.
"""

import requests
from bs4 import BeautifulSoup
import re

def diagnose_screener(symbol="RELIANCE"):
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml",
        "Accept-Language": "en-US,en;q=0.9",
    })

    urls = [
        f"https://www.screener.in/company/{symbol}/consolidated/?view=10years",
        f"https://www.screener.in/company/{symbol}/?view=10years",
    ]

    for url in urls:
        print(f"\nFetching: {url}")
        res = session.get(url, timeout=15)
        print(f"Status: {res.status_code}  |  Length: {len(res.text)}")
        if res.status_code != 200:
            continue

        soup = BeautifulSoup(res.text, "html.parser")

        # ── Print ALL section headings ──────────────────────────────────────
        print("\n=== ALL SECTIONS ===")
        for sec in soup.find_all("section"):
            h = sec.find("h2") or sec.find("h3")
            has_table = sec.find("table") is not None
            print(f"  heading: '{h.text.strip() if h else None}'  |  has_table: {has_table}")

        # ── Find balance sheet table ────────────────────────────────────────
        bal_table = None
        for sec in soup.find_all("section"):
            h = sec.find("h2") or sec.find("h3")
            if h and "balance" in h.text.lower():
                bal_table = sec.find("table")
                break

        if bal_table is None:
            print("\n!!! NO BALANCE SHEET TABLE FOUND !!!")
            continue

        # ── Print every raw row of the balance sheet table ─────────────────
        print("\n=== BALANCE SHEET TABLE — RAW ROWS ===")
        for i, row in enumerate(bal_table.find_all("tr")):
            cells = [c.get_text(strip=True) for c in row.find_all(["th", "td"])]
            print(f"  Row {i:02d}: {cells}")

        # ── Show what parse_table (old) would produce ───────────────────────
        print("\n=== parse_table RESULT (label → first 3 values) ===")
        rows_dict = {}
        headers = []
        for i, row in enumerate(bal_table.find_all("tr")):
            cells = [c.get_text(strip=True) for c in row.find_all(["th", "td"])]
            if not cells:
                continue
            if i == 0:
                headers = cells
                print(f"  HEADERS: {headers}")
                continue
            rows_dict[cells[0]] = cells[1:]

        for label, vals in rows_dict.items():
            print(f"  '{label}': {vals[:3]}")

        # ── Specifically look for equity-related rows ───────────────────────
        print("\n=== EQUITY-RELATED ROW SEARCH ===")
        equity_patterns = [
            "share capital", "equity share", "reserves", "other equity",
            "surplus", "net worth", "total equity", "shareholders",
            "stockholders", "networth"
        ]
        for label, vals in rows_dict.items():
            lc = label.lower()
            if any(p in lc for p in equity_patterns):
                print(f"  MATCH '{label}': {vals[:5]}")

        print("\n=== DONE ===")
        return  # stop after first successful URL

diagnose_screener("RELIANCE")


Fetching: https://www.screener.in/company/RELIANCE/consolidated/?view=10years
Status: 200  |  Length: 223260

=== ALL SECTIONS ===
  heading: 'None'  |  has_table: False
  heading: 'None'  |  has_table: False
  heading: 'Peer comparison'  |  has_table: False
  heading: 'Quarterly Results'  |  has_table: True
  heading: 'Profit & Loss'  |  has_table: True
  heading: 'Balance Sheet'  |  has_table: True
  heading: 'Cash Flows'  |  has_table: True
  heading: 'Ratios'  |  has_table: True
  heading: 'Insights'  |  has_table: True
  heading: 'Shareholding Pattern'  |  has_table: True
  heading: 'Documents'  |  has_table: False

=== BALANCE SHEET TABLE — RAW ROWS ===
  Row 00: ['', 'Mar 2015', 'Mar 2016', 'Mar 2017', 'Mar 2018', 'Mar 2019', 'Mar 2020', 'Mar 2021', 'Mar 2022', 'Mar 2023', 'Mar 2024', 'Mar 2025', 'Mar 2026']
  Row 01: ['Equity Capital', '2,943', '2,948', '2,959', '5,922', '5,926', '6,339', '6,445', '6,765', '6,766', '6,766', '13,532', '13,532']
  Row 02: ['Reserves', '215,556',

In [28]:
def get_date(row):
    q_map = {'Q1': '06-30', 'Q2': '09-30', 'Q3': '12-31', 'Q4': '03-31', 'FY': '03-31'}
    return pd.to_datetime(f"{row['year']}-{q_map[row['quarter']]}")

final_df['date'] = final_df.apply(get_date, axis=1)

In [29]:
final_df.head()

,symbol,year,quarter,net_income,total_assets,source,book_value,date
0,360ONE,2018,FY,380.0,9567.0,screener_html,1863.0,2018-03-31
1,360ONE,2019,FY,375.0,9763.0,screener_html,2911.0,2019-03-31
2,360ONE,2020,FY,201.0,13021.0,screener_html,2991.0,2020-03-31
3,360ONE,2021,FY,369.0,8739.0,screener_html,2828.0,2021-03-31
4,360ONE,2022,FY,578.0,10734.0,screener_html,3024.0,2022-03-31


In [30]:
final_df['date'] = pd.to_datetime(final_df['date'])

In [31]:
final_df = final_df.sort_values(['symbol', 'date'])

In [32]:
final_df = final_df.set_index('date')

In [33]:
fund_monthly = (
    final_df
    .groupby('symbol', group_keys=False)
    .resample('M')
    .ffill()
    .reset_index()
)

In [34]:
fund_monthly.head()

,date,symbol,year,quarter,net_income,total_assets,source,book_value
0,2018-03-31,360ONE,2018,FY,380.0,9567.0,screener_html,1863.0
1,2018-04-30,360ONE,2018,FY,380.0,9567.0,screener_html,1863.0
2,2018-05-31,360ONE,2018,FY,380.0,9567.0,screener_html,1863.0
3,2018-06-30,360ONE,2018,FY,380.0,9567.0,screener_html,1863.0
4,2018-07-31,360ONE,2018,FY,380.0,9567.0,screener_html,1863.0


In [35]:
cols_to_lag = ['net_income', 'total_assets', 'book_value']

fund_monthly[cols_to_lag] = (
    fund_monthly.groupby('symbol')[cols_to_lag]
    .shift(3)
)

In [36]:
# import sys
# !{sys.executable} -m pip install plotly

In [37]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'browser'   # safest across environments

# Step 1: Count + sort years
year_counts = (
    final_df['year']
    .astype(int)
    .value_counts()
    .sort_index()
    .reset_index()
)

year_counts.columns = ['year', 'count']

# Step 2: Plot
fig = px.bar(
    year_counts,
    x='year',
    y='count',
    title='Number of Data Points per Year',
    labels={'year': 'Year', 'count': 'Data Availability: Number of stocks out of 500'}
)

# Optional styling
fig.update_layout(
    xaxis=dict(dtick=1),   # show every year
    template='plotly_white'
)

fig.show()

In [38]:
final_df.shape

(4996, 7)

In [39]:
final_df.head()

,symbol,year,quarter,net_income,total_assets,source,book_value
date,,,,,,,
2018-03-31,360ONE,2018,FY,380.0,9567.0,screener_html,1863.0
2019-03-31,360ONE,2019,FY,375.0,9763.0,screener_html,2911.0
2020-03-31,360ONE,2020,FY,201.0,13021.0,screener_html,2991.0
2021-03-31,360ONE,2021,FY,369.0,8739.0,screener_html,2828.0
2022-03-31,360ONE,2022,FY,578.0,10734.0,screener_html,3024.0


In [40]:
final_df.to_csv(FIN_PROCESSED_DIR / "final_fin_data.csv")

In [41]:
price_data.head()

,date,"ã¯â»â¿""symbol""",series,prevclose,openprice,highprice,lowprice,lastprice,closeprice,averageprice,...,turnoverinrs,no.oftrades,deliverableqty,%dlyqttotradedqty,symbol,return,year,month,effective_year,turnover
1,2011-01-31,CUB,EQ,42.45,42.0,43.05,40.6,42.05,42.00,41.57,...,3.539672e+07,NaN,451643.0,53.05,CUB,NaN,2011,1,2010,3.575918e+07
2,2011-01-31,CEATLTD,EQ,111.1,110.75,113.5,110.0,113.5,112.05,112.12,...,1.379165e+07,NaN,56242.0,45.72,CEATLTD,NaN,2011,1,2010,1.378293e+07
3,2011-01-31,NAGARCONST,EQ,111.1,110.0,113.9,103.4,112.25,112.70,111.1,...,1.134332e+08,NaN,603046.0,59.07,NCC,NaN,2011,1,2010,1.150650e+08
4,2011-01-31,GLENMARK,EQ,309.75,305.0,312.7,296.0,307.05,307.15,301.03,...,1.629516e+08,NaN,297290.0,54.92,GLENMARK,NaN,2011,1,2010,1.662637e+08
5,2011-01-31,ABB,EQ,706.25,705.1,747.0,687.0,746.1,732.35,726.44,...,1.012008e+08,NaN,53216.0,38.2,ABB,NaN,2011,1,2010,1.020237e+08


In [42]:
price_data.columns = price_data.columns.str.replace('ã¯â»â¿"', '', regex=False)
price_data.columns = price_data.columns.str.strip()

In [43]:
# =====================================================================
# STEP 7: Derive Factor Inputs
# =====================================================================

In [44]:
# =====================================================================
# 7.1 Profitability
# =====================================================================

In [45]:
fund_monthly['profitability'] = (
    fund_monthly['net_income'] / fund_monthly['total_assets']
)

In [46]:
# =====================================================================
# 7.2 CMA (Investment)
# =====================================================================

In [47]:
fund_monthly['asset_growth'] = (
    fund_monthly.groupby('symbol')['total_assets']
    .pct_change()
)

In [48]:
rf_monthly.head()

,Rf_monthly
DATE,
2011-11-30,0.007315
2011-12-31,0.007290
2012-01-31,0.007342
2012-02-29,0.007110
2012-03-31,0.007692


In [49]:
rf_monthly = rf_monthly.reset_index()

In [50]:
rf_monthly.rename(columns={'DATE': 'date'}, inplace=True)

In [51]:
price_data['date'] = price_data['date'] + pd.offsets.MonthEnd(0)
fund_monthly['date'] = fund_monthly['date'] + pd.offsets.MonthEnd(0)

In [52]:
final = price_data.merge(fund_monthly, on=['symbol', 'date'], how='left')
final = final.merge(rf_monthly, on='date', how='left')

In [53]:
# ==========================================================
# Merge Book Value (FINAL STEP)
# ==========================================================

# Rename the duplicate column created by the fund_monthly merge if present
if 'book_value_x' in final.columns:
    final = final.drop(columns=[c for c in final.columns if c == 'book_value_y'])
    final = final.rename(columns={'book_value_x': 'book_value'})

print("Book value after merge:", final['book_value'].notna().sum())
print("Final shape:", final.shape)
print(final[['symbol', 'date', 'book_value']].dropna(subset=['book_value']).head(10))

Book value after merge: 30499
Final shape: (44361, 30)
         symbol       date  book_value
1156        ABB 2011-06-30      2534.0
1200  TATAELXSI 2011-06-30       182.0
1392  TATAELXSI 2011-07-31       182.0
1495        ABB 2011-07-31      2534.0
1672  TATAELXSI 2011-08-31       182.0
1777        ABB 2011-08-31      2534.0
1920        ABB 2011-09-30      2534.0
1930  TATAELXSI 2011-09-30       182.0
2051        ABB 2011-10-31      2534.0
2231  TATAELXSI 2011-10-31       182.0


In [54]:
final.head()

,date,"symbol""",series,prevclose,openprice,highprice,lowprice,lastprice,closeprice,averageprice,...,turnover,year_y,quarter,net_income,total_assets,source,book_value,profitability,asset_growth,Rf_monthly
0,2011-01-31,CUB,EQ,42.45,42.0,43.05,40.6,42.05,42.00,41.57,...,3.575918e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2011-01-31,CEATLTD,EQ,111.1,110.75,113.5,110.0,113.5,112.05,112.12,...,1.378293e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2011-01-31,NAGARCONST,EQ,111.1,110.0,113.9,103.4,112.25,112.70,111.1,...,1.150650e+08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2011-01-31,GLENMARK,EQ,309.75,305.0,312.7,296.0,307.05,307.15,301.03,...,1.662637e+08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2011-01-31,ABB,EQ,706.25,705.1,747.0,687.0,746.1,732.35,726.44,...,1.020237e+08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [55]:
final.shape

(44361, 30)

In [56]:
final.isnull().sum()

date                       0
symbol"                    0
series                     0
prevclose                  0
openprice                  0
highprice                  0
lowprice                   0
lastprice                  0
closeprice                 0
averageprice               0
totaltradedquantity        0
turnoverinrs               0
no.oftrades             1127
deliverableqty             0
%dlyqttotradedqty          0
symbol                     0
return                   244
year_x                     0
month                      0
effective_year             0
turnover                   0
year_y                 13144
quarter                13144
net_income             13862
total_assets           13862
source                 13144
book_value             13862
profitability          13862
asset_growth           14101
Rf_monthly              2980
dtype: int64

In [57]:
# =====================================================================
# 7.3 Market Cap
# =====================================================================

Market Cap = Price × Shares Outstanding

In [58]:
# Fetch shares
shares_map = {}

for sym in price_data['symbol'].unique():
    try:
        ticker = yf.Ticker(f"{sym}.NS")

        # Try multiple sources
        shares = ticker.info.get("sharesOutstanding", None)

        if shares is None:
            shares = ticker.get_shares_full(start="2015-01-01")

            if shares is not None and not shares.empty:
                shares = shares.iloc[-1]  # latest value

        shares_map[sym] = shares

    except:
        shares_map[sym] = None

# Map shares into final (not price_data)
final['shares_outstanding'] = final['symbol'].map(shares_map)

# Market cap
final['market_cap'] = final['closeprice'] * final['shares_outstanding']
final['market_cap'] = pd.to_numeric(final['market_cap'], errors='coerce')

# Debug
print("Non-null shares:", final['shares_outstanding'].notna().sum())
print("Non-null market cap:", final['market_cap'].notna().sum())

Non-null shares: 44361
Non-null market cap: 44361


In [59]:
print("Stocks with shares:", sum(pd.notna(list(shares_map.values()))))

Stocks with shares: 244


In [60]:
# =====================================================================
# 7.4 Book-to-Market
# =====================================================================

* Book Value = Shareholders' Equity (Net Worth)
* B/M = Book Value / Market Cap

In [61]:
final['bm'] = final['book_value'] / final['market_cap']

print("Final shape before filters:", final.shape)
print("Market cap non-null:", final['market_cap'].notna().sum())
print("Book value non-null:", final['book_value'].notna().sum())
print("BM non-null:", final['bm'].notna().sum())

Final shape before filters: (44361, 33)
Market cap non-null: 44361
Book value non-null: 30499
BM non-null: 30499


In [62]:
final = final[
    (final['market_cap'] > 0) &
    (final['book_value'] > 0)
]

final = final[final['bm'] > 0]

In [63]:
final.shape

(30091, 33)

In [64]:
print(final.columns)

Index(['date', 'symbol"', 'series', 'prevclose', 'openprice', 'highprice',
       'lowprice', 'lastprice', 'closeprice', 'averageprice',
       'totaltradedquantity', 'turnoverinrs', 'no.oftrades', 'deliverableqty',
       '%dlyqttotradedqty', 'symbol', 'return', 'year_x', 'month',
       'effective_year', 'turnover', 'year_y', 'quarter', 'net_income',
       'total_assets', 'source', 'book_value', 'profitability', 'asset_growth',
       'Rf_monthly', 'shares_outstanding', 'market_cap', 'bm'],
      dtype='object')


In [65]:
(final['year_x'] == final['year_y']).value_counts()

True     24829
False     5262
Name: count, dtype: int64

In [66]:
# ==========================================================
# FINAL COLUMN CLEANUP
# ==========================================================

# Remove corrupted NSE column
final = final.drop(columns=['symbol"'], errors='ignore')

# Remove duplicate accounting year
final = final.drop(columns=['year_y'], errors='ignore')

# Rename market year
final = final.rename(columns={'year_x': 'year'})

# Optional: reorder important columns
important_cols = [
    'date',
    'symbol',
    'year',
    'month',
    'return',
    'Rf_monthly',
    'market_cap',
    'bm',
    'profitability',
    'asset_growth'
]

remaining_cols = [
    c for c in final.columns
    if c not in important_cols
]

final = final[important_cols + remaining_cols]

---

## Phase 1 Summary — Data Ingestion & Cleaning ✓

### What was achieved

| Dataset | Coverage | Stored at |
|---|---|---|
| Price + volume (NSE EQ series) | 2011–present, ~470 stocks, monthly | `data/raw/*.csv` |
| Fundamentals: net income, total assets, book value | 2011–present, 476 stocks, annual (Screener) | `data/fin_data_processed/final_fin_data.csv` |
| Risk-free rate (91-day T-bill, FRED) | 2011–present, monthly compounded | `data/rf_india.csv` |

**Factor inputs derived:**
- `profitability` = Net Income / Total Assets → feeds **RMW**
- `asset_growth` = % change in Total Assets → feeds **CMA**
- `book_value` (lagged 3 months, annual → monthly expanded) → feeds **HML**
- `market_cap` = Price × Shares Outstanding → feeds **SMB** and **HML**
- `bm` = Book Value / Market Cap → feeds **HML**
- `return` and `Rf_monthly` → feed **MKT**

All fundamentals are lagged 3 months to prevent lookahead bias. Stocks with fewer than 70% monthly observations or median volume below 100k shares were dropped.

---

### How this feeds Phase 2 — Factor Construction

In Phase 2 we use `final` (the merged monthly panel) to construct all five Fama–French factors:

- **MKT** — value-weighted market return minus `Rf_monthly`, computed across all stocks each month
- **SMB** — monthly return spread between bottom and top tercile of `market_cap`, size-sorted portfolios
- **HML** — monthly return spread between top and bottom tercile of `bm`, annually rebalanced each July
- **RMW** — monthly return spread between top and bottom tercile of `profitability`
- **CMA** — monthly return spread between bottom and top tercile of `asset_growth` (low investment minus high)

Portfolio sorts follow the standard FF methodology: independent 2×3 sorts on size and each characteristic, rebalanced annually in July using prior fiscal year-end fundamentals (already lagged in Phase 1).